## Git pull (repo exists) or clone

In [3]:
from sagemaker import get_execution_role
role = get_execution_role()
print(role)  # should be arn:aws:iam::...:role/...

sagemaker.config INFO - Applied value from config key = SageMaker.PythonSDK.Modules.Session.DefaultS3Bucket
sagemaker.config INFO - Applied value from config key = SageMaker.PythonSDK.Modules.Session.DefaultS3ObjectKeyPrefix
arn:aws:iam::200915316557:role/service-role/AmazonSageMakerAdminIAMExecutionRole


In [14]:
# Cell 1: Clone or pull repository (SSH - no token needed!)
from pathlib import Path
%cd /mnt/custom-file-systems/s3/shared/
REPO_NAME = "ocr-agentic-rag"

if Path(REPO_NAME).exists():
    print(f"✓ Repository exists - pulling latest changes...")
    %cd {REPO_NAME}
    !git pull origin master
else:
    print(f"Cloning {REPO_NAME}...")
    !git clone git@github.com:leemingloon/ocr-agentic-rag.git
    %cd {REPO_NAME}

print("\n✓ Repository ready")
!git status

/mnt/custom-file-systems/s3/shared
Cloning ocr-agentic-rag...
Cloning into 'ocr-agentic-rag'...
remote: Enumerating objects: 100, done.
remote: Counting objects: 100% (100/100), done.
remote: Compressing objects: 100% (89/89), done.
remote: Total 100 (delta 12), reused 97 (delta 9), pack-reused 0 (from 0)
Receiving objects: 100% (100/100), 568.12 KiB | 164.00 KiB/s, done.
Resolving deltas: 100% (12/12), done.
Updating files: 100% (69/69), done.
/mnt/custom-file-systems/s3/shared/ocr-agentic-rag

✓ Repository ready
On branch master
Your branch is up to date with 'origin/master'.

nothing to commit, working tree clean


## Docker build and push (run if code changes were made)

In [1]:
import boto3
import sagemaker
from sagemaker import get_execution_role

sess = sagemaker.Session()
role = get_execution_role()                      # your SageMaker role
account_id = boto3.client('sts').get_caller_identity()['Account']
region = sess.boto_region_name                   # should be ap-southeast-2
ecr_repo_name = "ocr-agentic-rag-processing"     # create this repo in ECR if not exists
image_tag = "latest"

full_image_uri = f"{account_id}.dkr.ecr.{region}.amazonaws.com/{ecr_repo_name}:{image_tag}"

# Create ECR repo if missing
!aws ecr create-repository --repository-name {ecr_repo_name} --region {region} || echo "Repo exists"

# Docker login to ECR
!aws ecr get-login-password --region {region} | docker login --username AWS --password-stdin {account_id}.dkr.ecr.{region}.amazonaws.com

# Build (this is the long part — 10–25 min first time)
%cd /mnt/custom-file-systems/s3/shared/ocr-agentic-rag

# Clean any old failed build cache first (helps space)
!docker builder prune -f

# Build with the required network flag
!docker build --network sagemaker -t {full_image_uri} .

# Push
!docker push {full_image_uri}

print("Image pushed:", full_image_uri)

sagemaker.config INFO - Fetched defaults config from location: /etc/xdg/sagemaker/config.yaml
sagemaker.config INFO - Not applying SDK defaults from location: /home/sagemaker-user/.config/sagemaker/config.yaml
sagemaker.config INFO - Applied value from config key = SageMaker.PythonSDK.Modules.Session.DefaultS3Bucket
sagemaker.config INFO - Applied value from config key = SageMaker.PythonSDK.Modules.Session.DefaultS3ObjectKeyPrefix
sagemaker.config INFO - Applied value from config key = SageMaker.PythonSDK.Modules.Session.DefaultS3Bucket
sagemaker.config INFO - Applied value from config key = SageMaker.PythonSDK.Modules.Session.DefaultS3ObjectKeyPrefix
sagemaker.config INFO - Applied value from config key = SageMaker.PythonSDK.Modules.Session.DefaultS3Bucket
sagemaker.config INFO - Applied value from config key = SageMaker.PythonSDK.Modules.Session.DefaultS3ObjectKeyPrefix

An error occurred (RepositoryAlreadyExistsException) when calling the CreateRepository operation: The repository w

In [2]:
# Check if the image is valid
!docker images | grep ocr-agentic-rag-processing

200915316557.dkr.ecr.ap-southeast-2.amazonaws.com/ocr-agentic-rag-processing   latest    adc8bec35802   4 minutes ago   4.44GB


## Run processing job

In [7]:
import sagemaker
from sagemaker.processing import Processor, ProcessingInput, ProcessingOutput

role = sagemaker.get_execution_role()
bucket_name = "sagemaker-ocr-rag-mingloon"
full_image_uri = "200915316557.dkr.ecr.ap-southeast-2.amazonaws.com/ocr-agentic-rag-processing:latest"
print(role)  # should be arn:aws:iam::...:role/...

sagemaker.config INFO - Applied value from config key = SageMaker.PythonSDK.Modules.Session.DefaultS3Bucket
sagemaker.config INFO - Applied value from config key = SageMaker.PythonSDK.Modules.Session.DefaultS3ObjectKeyPrefix
arn:aws:iam::200915316557:role/service-role/AmazonSageMakerAdminIAMExecutionRole


In [8]:
from sagemaker.processing import Processor

processor = Processor(
    role=role,
    image_uri=full_image_uri,
    instance_count=1,
    instance_type="ml.t3.medium",           # cheap & sufficient
    max_runtime_in_seconds=3600,            # 1 hour max safety cap
    sagemaker_session=sagemaker.Session()
)

input_s3 = f"s3://{bucket_name}/input/credit_risk/uci_credit_default/"  # points to where your CSV is
output_s3 = f"s3://{bucket_name}/output/"

processor.run(
    inputs=[
        ProcessingInput(
            source=input_s3,
            destination="/opt/ml/processing/input",
            input_name="input_data"
        )
    ],
    outputs=[
        ProcessingOutput(
            source="/opt/ml/processing/output",
            destination=output_s3,
            output_name="results"
        )
    ],
    arguments=[
        "--mode", "sagemaker",
        "--s3-bucket", bucket_name,
        "--dry-run"  # dry-run remove this line later for full run
    ],
    wait=True,
    logs=True
)

print("Job completed. Check outputs in S3:", output_s3)
print("Job name (for console):", processor._current_job_name)

sagemaker.config INFO - Applied value from config key = SageMaker.PythonSDK.Modules.Session.DefaultS3Bucket
sagemaker.config INFO - Applied value from config key = SageMaker.PythonSDK.Modules.Session.DefaultS3ObjectKeyPrefix
...............⚠ PaddleOCR not available
ℹ️  DRY-RUN mode → continuing without Anthropic key
🧪 DRY-RUN MODE: No API calls will be made (testing pipeline only)
SageMaker Evaluation with S3 Integration
✓ S3 client initialized: s3://sagemaker-ocr-rag-mingloon
⚠ Downloading reference data from S3: s3://sagemaker-ocr-rag-mingloon/data/reference/training_data.csv
✓ Reference data downloaded and cached
✓ Reference data loaded: 30000 samples, 25 features
1. Loading reference data from S3...
✓ Reference data loaded: 30000 samples
2. Running credit risk evaluation...
⚠ Downloading FinBERT from HuggingFace: ProsusAI/finbert
#015Loading weights:   0%|          | 0/201 [00:00<?, ?it/s]#015Loading weights:   0%|          | 1/201 [00:00<00:00, 4888.47it/s, Materializing param=ber